# TESTING Save a European map of BMR on SHERPA resolution using SHERPA country map

In [1]:
import glob
import os
import xarray as xr
import numpy as np
import config
from utils.utils import require_dir
import pathlib

In [2]:
# === Path config ===
EU_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SHERPA" / "processed")
BMR_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR")
MASKS_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SHERPA" / "masks")

In [7]:
# === Health variables ===
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER",
               "STROKE", "DEMENTIA"]

In [8]:
# === Set GBD version ===
GBD_version = "GBD23"

In [3]:
# Load country masks
in_file = "country_mask_all.nc"
in_path = os.path.join(MASKS_DIR, in_file)
mask = xr.open_dataarray(in_path)

In [4]:
# Load one SHERPA file for lat/lon target grid
eu_file = "EU_concentration_H_2040.nc"
eu_path = os.path.join(EU_DIR, eu_file)
eu = xr.open_dataarray(eu_path)

# Rename to match eu's dim names if needed
if "latitude" in mask.dims and "latitude" not in eu.dims:
    eu = eu.rename({"latitude": "lat", "longitude": "lon"})
elif "lat" in mask.dims and "lat" not in eu.dims:
    mask = mask.rename({"lat": "latitude", "lon": "longitude"})

# Verify grids actually match before assuming alignment works —
# these masks were built on "the SHERPA grid" per their file metadata,
# but confirm rather than trust that
lat_name = "latitude" if "latitude" in mask.dims else "lat"
lon_name = "longitude" if "longitude" in mask.dims else "lon"

same_shape = mask.sizes[lat_name] == eu.sizes[lat_name] and mask.sizes[lon_name] == eu.sizes[lon_name]
if same_shape:
    lat_ok = np.allclose(mask[lat_name].values, eu[lat_name].values, atol=1e-4)
    lon_ok = np.allclose(mask[lon_name].values, eu[lon_name].values, atol=1e-4)
else:
    lat_ok = lon_ok = False

print(f"Same shape: {same_shape}, lat aligned: {lat_ok}, lon aligned: {lon_ok}")

if not (lat_ok and lon_ok):
    # Grids are close but not bit-identical (or different extents) — reindex
    # rather than assume a plain multiply will align correctly
    mask = mask.reindex({lat_name: eu[lat_name], lon_name: eu[lon_name]},
                         method="nearest", tolerance=1e-4)
    n_missing = int(mask.isnull().any(dim="country").sum())
    print(f"Reindexed mask to EU grid. Cells with no match within tolerance: {n_missing}")
else:
    print("Mask already on EU grid — no regridding needed.")

Same shape: True, lat aligned: True, lon aligned: True
Mask already on EU grid — no regridding needed.


In [5]:
total_coverage = mask.sum(dim="country")

# Real overlap of fractional masks at coastlines/borders is expected and fine.
# What's NOT fine is coverage that meaningfully exceeds 1 (double-counted area)
n_overclaimed = int((total_coverage > 1.05).sum())
print(f"Cells with coverage > 1.05 (likely a mask construction issue, not normal overlap): {n_overclaimed}")
if n_overclaimed > 0:
    print("Inspect these before trusting area-weighted sums — coverage above 1 "
          "means some area is being double-counted across countries.")

Cells with coverage > 1.05 (likely a mask construction issue, not normal overlap): 0


In [6]:
def build_value_grid_weighted(bmr, mask, quantile="mean"):
    """
    Area-weight BMR values across countries at every cell, using the
    fractional mask as weights. A cell that's 60% France / 40% Germany
    gets 0.6*BMR_France + 0.4*BMR_Germany, rather than a hard winner-take-all
    assignment.
    """
    bmr_vals = bmr.sel(quantile=quantile)

    # Align by country label, not position — check for naming mismatches first
    mask_countries = set(mask.country.values)
    bmr_countries = set(bmr_vals.country.values)

    missing_in_bmr = mask_countries - bmr_countries
    if missing_in_bmr:
        print(f"NOTE: {len(missing_in_bmr)} mask countries have no BMR match "
              f"(their contribution will be 0, not NaN — fine if outside Europe, "
              f"a problem if it's a naming mismatch like 'Czechia' vs 'Czech Republic'): "
              f"{sorted(missing_in_bmr)[:10]}{'...' if len(missing_in_bmr) > 10 else ''}")

    # Reindex bmr onto mask's country coordinate; unmatched countries -> NaN -> 0 weight contribution
    bmr_aligned = bmr_vals.reindex(country=mask.country)
    bmr_filled = bmr_aligned.fillna(0)

    weighted = (mask * bmr_filled).sum(dim="country")

    # Where total coverage is 0 (ocean, no country), result should be NaN, not 0 —
    # a BMR of 0 there would be misleading in the mortality calc downstream
    total_coverage = mask.sum(dim="country")
    weighted = weighted.where(total_coverage > 0)

    weighted.name = f"BMR_weighted_{quantile}"
    return weighted

In [9]:
for health_VAR in health_vars[:1]:
    bmr_file = f"{GBD_version}_BMR_Country_{health_VAR}_newlabels_2015-2019.nc"
    bmr_path = os.path.join(BMR_DIR, bmr_file)
    bmr = xr.open_dataarray(bmr_path)

    bmr_mean_grid = build_value_grid_weighted(bmr, mask, "mean")
    bmr_lower_grid = build_value_grid_weighted(bmr, mask, "lower")
    bmr_upper_grid = build_value_grid_weighted(bmr, mask, "upper")

    result = xr.Dataset({
        "BMR_mean": bmr_mean_grid,
        "BMR_lower": bmr_lower_grid,
        "BMR_upper": bmr_upper_grid,
    })

    expected_shape = (eu.sizes[lat_name], eu.sizes[lon_name])
    assert result.BMR_mean.shape == expected_shape, "Shape mismatch vs EU target grid!"
    print("Output shape:", result.BMR_mean.shape, "-- matches EU grid:", expected_shape)

    out_file = f"{GBD_version}_BMR_European_Country_Map_{health_VAR}_2015-2019.nc"
    out_path = os.path.join(BMR_DIR, out_file)
    print(f"Saving to {out_path}")
    result.to_netcdf(out_path)

print("All processing complete.")

NOTE: 13 mask countries have no BMR match (their contribution will be 0, not NaN — fine if outside Europe, a problem if it's a naming mismatch like 'Czechia' vs 'Czech Republic'): ['Aland', 'Czechia', 'Faroe Islands', 'Gibraltar', 'Guernsey', 'Isle of Man', 'Jersey', 'Kosovo', 'Liechtenstein', 'North Macedonia']...
NOTE: 13 mask countries have no BMR match (their contribution will be 0, not NaN — fine if outside Europe, a problem if it's a naming mismatch like 'Czechia' vs 'Czech Republic'): ['Aland', 'Czechia', 'Faroe Islands', 'Gibraltar', 'Guernsey', 'Isle of Man', 'Jersey', 'Kosovo', 'Liechtenstein', 'North Macedonia']...
NOTE: 13 mask countries have no BMR match (their contribution will be 0, not NaN — fine if outside Europe, a problem if it's a naming mismatch like 'Czechia' vs 'Czech Republic'): ['Aland', 'Czechia', 'Faroe Islands', 'Gibraltar', 'Guernsey', 'Isle of Man', 'Jersey', 'Kosovo', 'Liechtenstein', 'North Macedonia']...


MergeError: conflicting values for variable 'quantile' on objects to be combined. You can skip this check by specifying compat='override'.

In [11]:
mask.country

<xarray.DataArray 'country' (country: 67)> Size: 6kB
array(['Andorra', 'Albania', 'Armenia', 'Austria', 'Aland', 'Azerbaijan',
       'Bosnia and Herzegovina', 'Belgium', 'Bulgaria', 'Belarus',
       'Switzerland', 'Cyprus', 'Czechia', 'Germany', 'Denmark', 'Algeria',
       'Estonia', 'Spain', 'Finland', 'Faroe Islands', 'France',
       'United Kingdom', 'Georgia', 'Guernsey', 'Gibraltar', 'Greece',
       'Croatia', 'Hungary', 'Ireland', 'Israel', 'Isle of Man', 'Iraq',
       'Iran', 'Iceland', 'Italy', 'Jersey', 'Jordan', 'Lebanon',
       'Liechtenstein', 'Lithuania', 'Luxembourg', 'Latvia', 'Libya',
       'Morocco', 'Monaco', 'Moldova', 'Montenegro', 'North Macedonia',
       'Malta', 'Netherlands', 'Norway', 'Poland', 'Palestine', 'Portugal',
       'Romania', 'Republic of Serbia', 'Russia', 'Sweden', 'Slovenia',
       'Slovakia', 'San Marino', 'Syria', 'Tunisia', 'Turkey', 'Ukraine',
       'Vatican', 'Kosovo'], dtype='<U22')
Coordinates:
  * country  (country) <U22 6kB 'Andorra' 'Albania' ... 'Vatican' 'Kosovo'

In [12]:
countries_1 = set(bmr["country"].values)  # Country labels we want
countries_2 = set(mask["country"].values)  # Country labels we need to change

# Show list of country names we will change FROM
mismatched = countries_2 - countries_1
print("Mismatched country names (from):", sorted(mismatched))

# Show list of country names we will change TO
mismatched = countries_1 - countries_2
print("Mismatched country names (to):", sorted(mismatched))

Mismatched country names (from): ['Aland', 'Czechia', 'Faroe Islands', 'Gibraltar', 'Guernsey', 'Isle of Man', 'Jersey', 'Kosovo', 'Liechtenstein', 'North Macedonia', 'Republic of Serbia', 'Russia', 'Vatican']
Mismatched country names (to): ['Afghanistan', 'American Samoa', 'Angola', 'Antigua and Barbuda', 'Argentina', 'Australia', 'Bahrain', 'Bangladesh', 'Barbados', 'Belize', 'Benin', 'Bermuda', 'Bhutan', 'Bolivia', 'Botswana', 'Brazil', 'Brunei', 'Burkina Faso', 'Burundi', 'Cambodia', 'Cameroon', 'Canada', 'Cape Verde', 'Central African Republic', 'Chad', 'Chile', 'China', 'Colombia', 'Comoros', 'Congo', 'Cook Islands', 'Costa Rica', "Cote d'Ivoire", 'Cuba', 'Czech Republic', 'Democratic Republic of the Congo', 'Djibouti', 'Dominica', 'Dominican Republic', 'Ecuador', 'Egypt', 'El Salvador', 'Equatorial Guinea', 'Eritrea', 'Ethiopia', 'Federated States of Micronesia', 'Fiji', 'Gabon', 'Ghana', 'Greenland', 'Grenada', 'Guam', 'Guatemala', 'Guinea', 'Guinea-Bissau', 'Guyana', 'Haiti', 